# Backpropagation: what it is
The 2-2-1 regression network of the Note (linear activations), the first student, the 9 derivatives from the
chain rule, and one gradient-descent update. The derivatives are checked numerically and with TensorFlow's
GradientTape.

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"      # hide TensorFlow's start-up messages
import numpy as np
import pandas as pd

# four students: CGPA, profile score (both out of 10) and package in LPA
df = pd.DataFrame([[8, 8, 4], [7, 9, 5], [6, 10, 6], [5, 12, 7]],
                  columns=["cgpa", "profile_score", "lpa"])
df.to_csv("data/students.csv", index=False)
df

## Step 0: initialise. All weights 0.1, all biases 0.
`W1[i-1, j-1]` is $W^1_{ij}$ (from input i to hidden node j); `W2[i-1]` is $W^2_{i1}$.

In [ ]:
def init_params():
    return {"W1": np.full((2, 2), 0.1), "b1": np.zeros(2), "W2": np.full(2, 0.1), "b2": 0.0}

p = init_params()
print(sum(np.size(v) for v in p.values()), "trainable parameters")   # 9

## Steps 1-3: pick a student, predict (forward propagation), compute the loss

In [ ]:
def forward(x, p):
    O1 = p["W1"].T @ x + p["b1"]          # outputs O11, O12 (linear activation)
    y_hat = p["W2"] @ O1 + p["b2"]        # output O21 = y_hat
    return y_hat, O1

x, y = df.loc[0, ["cgpa", "profile_score"]].to_numpy(float), float(df.loc[0, "lpa"])
y_hat, O1 = forward(x, p)
loss = (y - y_hat) ** 2
print("O11, O12 =", O1, " y_hat =", round(y_hat, 4), " loss =", round(loss, 4))

## The 9 derivatives from the chain rule

In [ ]:
def gradients(x, y, y_hat, O1, p):
    g = -2 * (y - y_hat)                  # dL/dy_hat, shared by all nine
    return {
        "W2": g * O1,                     # dL/dW2_11 = g*O11, dL/dW2_21 = g*O12
        "b2": g,                          # dL/db21 = g
        # dL/dW1_ij = g * W2_j1 * x_i ; dL/db1j = g * W2_j1
        "W1": np.outer(x, g * p["W2"]),
        "b1": g * p["W2"],
    }

grads = gradients(x, y, y_hat, O1, p)
for k, v in grads.items():
    print(k, np.round(v, 4))

## Check 1: nudge each parameter by a tiny h and watch the loss (finite differences)

In [ ]:
def loss_of(p):
    return (y - forward(x, p)[0]) ** 2

h = 1e-6
for name in p:
    arr = np.atleast_1d(np.array(p[name], dtype=float))
    num = np.zeros_like(arr)
    for idx in np.ndindex(arr.shape):
        q = {k: np.array(v, dtype=float) for k, v in p.items()}
        qa = np.atleast_1d(q[name]); qa[idx] += h
        q[name] = qa if np.ndim(p[name]) else qa[0]
        num[idx] = (loss_of(q) - loss) / h
    print(name, num.round(3))

## Check 2: TensorFlow's automatic differentiation (GradientTape)

In [ ]:
import tensorflow as tf

tp = {k: tf.Variable(np.array(v, dtype=np.float64)) for k, v in p.items()}
with tf.GradientTape() as tape:
    O1_tf = tf.linalg.matvec(tf.transpose(tp["W1"]), x) + tp["b1"]
    y_hat_tf = tf.reduce_sum(tp["W2"] * O1_tf) + tp["b2"]
    loss_tf = (y - y_hat_tf) ** 2
tape_grads = tape.gradient(loss_tf, tp)
for k in p:
    print(k, np.round(tape_grads[k].numpy(), 4), np.allclose(tape_grads[k].numpy(), grads[k]))

## Step 4: one gradient-descent update, learning rate 0.001

In [ ]:
lr = 0.001
p_new = {k: p[k] - lr * grads[k] for k in p}
for k, v in p_new.items():
    print(k, np.round(v, 6))
y_hat2, _ = forward(x, p_new)
print("new prediction", round(y_hat2, 4), "new loss", round((y - y_hat2) ** 2, 4))

## One epoch: all four students, one update each

In [ ]:
p_ep = init_params()
for i in range(4):
    xi, yi = df.loc[i, ["cgpa", "profile_score"]].to_numpy(float), float(df.loc[i, "lpa"])
    yh, O = forward(xi, p_ep)
    print(f"student {i+1}: y_hat {yh:.3f}, loss {(yi - yh) ** 2:.3f}")
    gr = gradients(xi, yi, yh, O, p_ep)
    p_ep = {k: p_ep[k] - lr * gr[k] for k in p_ep}